In [4]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [5]:
# ─────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────
LABEL_COL   = "language"           # change if Member A used a different column name
TEST_SIZE   = 0.20
RANDOM_SEED = 42
OUTPUT_DIR  = "."                  # folder where .pkl files + cv_results.csv are saved
CV_FOLDS    = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [7]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("Loading CSVs …")
raw_df      = pd.read_csv("features_raw.csv")
selected_df = pd.read_csv("features_selected.csv")

datasets = {
    "raw":      raw_df,
    "selected": selected_df,
}

Loading CSVs …


In [8]:
# ─────────────────────────────────────────────
# 2. PREPROCESSING HELPER
# ─────────────────────────────────────────────
def preprocess(df, label_col=LABEL_COL):
    """Returns X_train, X_test, y_train, y_test, label_encoder."""
    df = df.dropna()

    le = LabelEncoder()
    y  = le.fit_transform(df[label_col].astype(str))
    X  = df.drop(columns=[label_col]).select_dtypes(include=[np.number])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test, le

In [9]:
# ─────────────────────────────────────────────
# 3. MODEL DEFINITIONS + HYPERPARAMETER GRIDS
# ─────────────────────────────────────────────
# Each entry: (model_key, estimator_inside_pipeline, param_grid_with_"clf__"_prefix)
model_configs = [
    (
        "SVM",
        SVC(random_state=RANDOM_SEED, probability=True),
        {
            "clf__C":      [0.1, 1, 10, 100],
            "clf__gamma":  ["scale", "auto"],
            "clf__kernel": ["rbf", "linear"],
        },
    ),
    (
        "RandomForest",
        RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
        {
            "clf__n_estimators":   [100, 200],
            "clf__max_depth":      [None, 10, 20],
            "clf__min_samples_split": [2, 5],
        },
    ),
    (
        "LogisticRegression",
        LogisticRegression(random_state=RANDOM_SEED, max_iter=1000),
        {
            "clf__C":       [0.01, 0.1, 1, 10],
            "clf__solver":  ["lbfgs", "saga"],
            "clf__penalty": ["l2"],
        },
    ),
]

In [10]:
# ─────────────────────────────────────────────
# 4. TRAINING LOOP
# ─────────────────────────────────────────────
cv_records  = []          # rows for cv_results.csv
best_models = {}          # {model_key: best_estimator} from "selected" feature set

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

for dataset_name, df in datasets.items():
    print(f"\n{'='*55}")
    print(f"  Feature set : {dataset_name}")
    print(f"{'='*55}")

    X_train, X_test, y_train, y_test, le = preprocess(df)

    for model_name, estimator, param_grid in model_configs:
        print(f"\n  Training {model_name} …")

        # Build a Pipeline: scaler → model
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    estimator),
        ])

        grid_search = GridSearchCV(
            pipe,
            param_grid,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1,
            verbose=0,
            refit=True,          # refit best model on full training set
            return_train_score=True,
        )
        grid_search.fit(X_train, y_train)

        best_params  = grid_search.best_params_
        mean_cv_acc  = grid_search.best_score_
        std_cv_acc   = grid_search.cv_results_["std_test_score"][grid_search.best_index_]
        test_acc     = grid_search.best_estimator_.score(X_test, y_test)

        print(f"    Best params : {best_params}")
        print(f"    CV accuracy : {mean_cv_acc:.4f} ± {std_cv_acc:.4f}")
        print(f"    Test accuracy: {test_acc:.4f}")

        # Record results
        record = {
            "feature_set":   dataset_name,
            "model":         model_name,
            "mean_cv_acc":   round(mean_cv_acc, 4),
            "std_cv_acc":    round(std_cv_acc,  4),
            "test_acc":      round(test_acc,    4),
            "best_params":   str(best_params),
            "n_train":       len(X_train),
            "n_test":        len(X_test),
            "n_features":    X_train.shape[1],
        }
        cv_records.append(record)

        # Save model trained on "selected" features (primary deliverable for Member C)
        if dataset_name == "selected":
            best_models[model_name] = grid_search.best_estimator_


  Feature set : raw

  Training SVM …
    Best params : {'clf__C': 10, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
    CV accuracy : 0.9255 ± 0.0088
    Test accuracy: 0.9623

  Training RandomForest …
    Best params : {'clf__max_depth': 20, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}
    CV accuracy : 0.9015 ± 0.0220
    Test accuracy: 0.9144

  Training LogisticRegression …
    Best params : {'clf__C': 1, 'clf__penalty': 'l2', 'clf__solver': 'saga'}
    CV accuracy : 0.8767 ± 0.0171
    Test accuracy: 0.8938

  Feature set : selected

  Training SVM …
    Best params : {'clf__C': 10, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
    CV accuracy : 0.9307 ± 0.0087
    Test accuracy: 0.9555

  Training RandomForest …
    Best params : {'clf__max_depth': None, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}
    CV accuracy : 0.8904 ± 0.0236
    Test accuracy: 0.8973

  Training LogisticRegression …
    Best params : {'clf__C': 10, 'clf__penalty': 'l2', 'clf__solver': 

In [11]:
# ─────────────────────────────────────────────
# 5. SAVE TRAINED MODELS  (.pkl)
# ─────────────────────────────────────────────
print("\n\nSaving models …")
model_file_map = {
    "SVM":               "svm_model.pkl",
    "RandomForest":      "rf_model.pkl",
    "LogisticRegression":"lr_model.pkl",
}

for model_name, filename in model_file_map.items():
    if model_name in best_models:
        path = os.path.join(OUTPUT_DIR, filename)
        joblib.dump(best_models[model_name], path)
        print(f"  Saved {filename}")
    else:
        print(f"  WARNING: {model_name} not found – skipping save.")



Saving models …
  Saved svm_model.pkl
  Saved rf_model.pkl
  Saved lr_model.pkl


In [12]:
# ─────────────────────────────────────────────
# 6. SAVE cv_results.csv
# ─────────────────────────────────────────────
cv_df = pd.DataFrame(cv_records)

# Reorder columns for readability
col_order = [
    "model", "feature_set",
    "mean_cv_acc", "std_cv_acc", "test_acc",
    "n_train", "n_test", "n_features",
    "best_params",
]
cv_df = cv_df[col_order]

results_path = os.path.join(OUTPUT_DIR, "cv_results.csv")
cv_df.to_csv(results_path, index=False)
print(f"\nSaved cv_results.csv  ({len(cv_df)} rows)")



Saved cv_results.csv  (6 rows)


In [13]:
# ─────────────────────────────────────────────
# 7. QUICK SUMMARY TABLE
# ─────────────────────────────────────────────
print("\n" + "="*55)
print("  SUMMARY")
print("="*55)
summary = cv_df[["model", "feature_set", "mean_cv_acc", "std_cv_acc", "test_acc"]]
print(summary.to_string(index=False))

print("\nDone! Hand off to Member C:")
print("  svm_model.pkl")
print("  rf_model.pkl")
print("  lr_model.pkl")
print("  cv_results.csv")



  SUMMARY
             model feature_set  mean_cv_acc  std_cv_acc  test_acc
               SVM         raw       0.9255      0.0088    0.9623
      RandomForest         raw       0.9015      0.0220    0.9144
LogisticRegression         raw       0.8767      0.0171    0.8938
               SVM    selected       0.9307      0.0087    0.9555
      RandomForest    selected       0.8904      0.0236    0.8973
LogisticRegression    selected       0.8639      0.0115    0.8733

Done! Hand off to Member C:
  svm_model.pkl
  rf_model.pkl
  lr_model.pkl
  cv_results.csv
